In [7]:
from langgraph.graph import StateGraph, START, END
from RAG.types import state
from RAG.llm.model import get_OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
llm = get_OpenAI()
output_parser = StrOutputParser()

In [3]:
from RAG.tools.call_Vehicle_tools import get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest, State, Vehicle

In [4]:
print(f"도구 이름: {get_brand_list_tool.name}")
print(f"도구 설명: {get_brand_list_tool.description}")

도구 이름: get_brand_list
도구 설명: 자동차 제조사 목록을 조회하는 함수입니다.

Args:
    headers (dict): API 호출 시 필요한 헤더 정보.

Returns:
    dict: 제조사 목록 JSON 데이터.


In [5]:
tools = [get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest]

In [8]:
tool_node = ToolNode(tools)

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

access_token = os.getenv("CODEF_ACCESS_TOKEN")

In [11]:
from pydantic import BaseModel
from typing_extensions import Annotated, Sequence, TypedDict
from langchain.schema import BaseMessage

In [14]:
headers: headers={
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

State: State = {
    "messages": [{"role": "user", "human": "브랜드 조회해줘."}],
    "headers":headers,
    "vehicle":Vehicle(),
    "ask_human": bool
}
# config 설정
config = {"configurable": {"thread_id": "1"}}

In [18]:
State : State = {
    "messages": [{"role": "user", "content": "브랜드 조회해줘."}],
    "headers" : headers,
    "vehicle" : Vehicle(),
    "ask_human": False
}

In [19]:
State["messages"]

[{'role': 'user', 'content': '브랜드 조회해줘.'}]

In [22]:
import requests
import json
import urllib.parse
def get_brand_list_tool(State: dict) -> dict:
    """
    자동차 제조사 목록을 조회하는 함수입니다.

    Args:
        headers (dict): API 호출 시 필요한 헤더 정보.

    Returns:
        dict: 제조사 목록 JSON 데이터.
    """
    headers = State["headers"] # headers 추출
    url = "https://api.codef.io/v1/kr/car/brand-list"
    response = requests.get(url=url, headers=headers)
    
    if response.status_code == 200:
        try:
            decoded_response = urllib.parse.unquote(response.text)
            return json.loads(decoded_response)
        except json.JSONDecodeError:
            return {"error": "JSON 변환 실패"}
    else:
        return {"error": f"API 요청 실패: {response.status_code}"}

In [23]:
get_brand_list_tool(State)

{'result': {'code': 'CF-00000', 'extraMessage': '', 'message': '성공'},
 'data': [{'cd': 'AD', 'nm': 'AD'},
  {'cd': 'CT&T', 'nm': 'CT&T'},
  {'cd': '기아', 'nm': '기아'},
  {'cd': '기타', 'nm': '기타'},
  {'cd': '대우', 'nm': '대우'},
  {'cd': '대창', 'nm': '대창'},
  {'cd': '삼성', 'nm': '삼성'},
  {'cd': '쌍용', 'nm': '쌍용'},
  {'cd': '외산', 'nm': '외산'},
  {'cd': '현EV', 'nm': '현EV'},
  {'cd': '현대', 'nm': '현대'}]}